# Lab 2 — Ciclo de treino, vazamento e métricas
**João Rietra · Turma BD-P4-2026.2 · Individual · 1,0 ponto na AV1**

Este notebook responde ao enunciado do Lab 2 com a base Red Wine Quality. Ele é **explicativo**: o código avaliado mora no pacote `lab02lib/` (ao lado deste arquivo) e as células chamam essas funções com poucas linhas. O que importa aqui é o porquê de cada passo e o que cada saída mostra.

Os números citados nos textos depois de cada resultado são gerados pelo código a partir dos arquivos de `resultados/`. Assim o texto não contradiz a saída quando o notebook é executado de novo.

Ordem: preparação → B2 → B1 → A1–A4 → C → D7 → D4 → D5/D6 → D1–D3 → D8 → conferência.

## 0. Ambiente e pasta
Abra este notebook dentro da pasta `lab-02/`, onde estão `apoio.py`, `dados/` e `lab02lib/`. CPU é suficiente; o dispositivo é fixo em `'cpu'`.

In [ ]:
from pathlib import Path
import sys, copy, time, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
from IPython.display import display, Markdown
from apoio import FEATURES, carregar_base, fixar_semente, criar_modelo
from apoio import treinar_epoca, avaliar_perda, obter_escores
from lab02lib import MeuDataset, particionar_sem_vazar, treinar_com_parada

SEED = 42
fixar_semente(SEED)
DISPOSITIVO = 'cpu'
RESULTADOS = Path('resultados')
RESULTADOS.mkdir(exist_ok=True)
print('Python:', sys.version.split()[0])
for pacote in ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'torch']:
    print(pacote, importlib.metadata.version(pacote))

### Leitura e conferência da base
`carregar_base` lê o CSV com `;`, confere sua integridade e cria o alvo e os identificadores de controle. As únicas entradas permitidas estão em `FEATURES`. Não aprende média, desvio, pesos ou partições.

Ver as contagens do arquivo não é treinar um modelo. Depois da B2, o teste principal fica fora das decisões e dos experimentos de erro.

In [ ]:
base = carregar_base()
print(base.head().to_string(index=False))
print('\nLinhas:', len(base), '| características:', len(FEATURES))
print('Classe positiva:', int(base.alvo.sum()), '| proporção:', base.alvo.mean())
print('Perfis distintos:', base.grupo_perfil.nunique())
assert set(FEATURES).isdisjoint({'quality', 'alvo', 'id_linha', 'grupo_perfil'})

### Exemplo mínimo: o que faz uma atualização
O bloco a seguir usa **quatro observações artificiais com duas entradas**, apenas para conferir instalação e formas. Não usa Wine Quality, não mede generalização e não atende às partes avaliadas. Um logit é transformado em escore por sigmoide só ao obter previsões; a perda recebe o logit.

In [ ]:
x_exemplo = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_exemplo = torch.tensor([0., 0., 0., 1.])
exemplo_loader = DataLoader(TensorDataset(x_exemplo, y_exemplo), batch_size=2)
exemplo_modelo = criar_modelo(2).to(DISPOSITIVO)
exemplo_otimizador = torch.optim.Adam(exemplo_modelo.parameters(), lr=0.001)
exemplo_perda = treinar_epoca(exemplo_modelo, exemplo_loader, exemplo_otimizador,
                              nn.BCEWithLogitsLoss(), DISPOSITIVO)
print('Teste de ambiente concluído. Perda de treino artificial:', exemplo_perda)
del exemplo_modelo, exemplo_loader, exemplo_otimizador
fixar_semente(SEED)

### Mapa do código
Cada implementação avaliada está em um arquivo do pacote `lab02lib/`. O `apoio.py` do professor não foi modificado.

| Item do enunciado | O que implementa | Arquivo |
|---|---|---|
| B2 | `particionar_sem_vazar`, `auditar_particoes` | [lab02lib/split.py](lab02lib/split.py) |
| A1, A2 | `MeuDataset`, `criar_datasets`, `criar_loaders`, verificações | [lab02lib/data.py](lab02lib/data.py) |
| A3 | `medir_tempos`, `resumir_tempos`, `grafico_tempos` | [lab02lib/timing.py](lab02lib/timing.py) |
| A4 | `demonstrar_batchnorm` | [lab02lib/batchnorm.py](lab02lib/batchnorm.py) |
| C (formas 1 e 4) | experimentos incorreto × corrigido, 5 sementes | [lab02lib/leakage.py](lab02lib/leakage.py) |
| D7 | `treinar_com_parada`, prova de restauração | [lab02lib/train.py](lab02lib/train.py) |
| D4, D5, D6 | custos, busca de limiar, fórmula, calibração, congelamento | [lab02lib/decision.py](lab02lib/decision.py) |
| D1, D2, D3 | métricas, baseline, curvas ROC e PR | [lab02lib/evaluate.py](lab02lib/evaluate.py) |
| MLflow (registro opcional) | rastreio dos experimentos em `mlflow.db` | [lab02lib/tracking.py](lab02lib/tracking.py) |

## B2. Separe antes de normalizar
**Pergunta de uso:** selecionar amostras de vinho tinto de nota alta usando só as 11 medidas físico-químicas.

**Estratégia:** `GroupShuffleSplit` por `grupo_perfil`, duas vezes. A primeira reserva 20% dos grupos para o teste. A segunda tira 25% dos grupos que sobraram para a validação (0,25 de 0,8 = 0,2 do total). Perfis idênticos ficam sempre no mesmo conjunto. A semente 42 foi declarada antes de qualquer modelo e não foi procurada por métrica.

A auditoria confere: contagens, cobertura das 1599 linhas, `id_linha` sem sobreposição, zero perfis compartilhados entre cada par e as duas classes em cada conjunto. Se qualquer regra falhar, a função levanta erro.

In [ ]:
from lab02lib.split import auditar_particoes

particoes = particionar_sem_vazar(base, SEED)
b2_tabela, b2_pares = auditar_particoes(particoes, base)
b2_tabela.to_csv(RESULTADOS / 'b2_auditoria.csv', index=False)
b2_pares.to_csv(RESULTADOS / 'b2_pares.csv', index=False)
display(b2_tabela.round(4))
display(b2_pares)
print('OK: cobertura total das', len(base), 'linhas; id_linha disjunto; 0 perfis compartilhados; as duas classes em cada conjunto.')

In [ ]:
t = b2_tabela.set_index('conjunto')
maior = t.desvio_pp_vs_alvo.abs().max()
display(Markdown(f'''**Como ler.** Treino, validação e teste ficaram com {t.linhas['treino']}, {t.linhas['validacao']} e {t.linhas['teste']} linhas ({100*t.proporcao_linhas['treino']:.1f}%, {100*t.proporcao_linhas['validacao']:.1f}% e {100*t.proporcao_linhas['teste']:.1f}%). O sorteio é por perfil, então as proporções de linhas só se aproximam de 60/20/20. O maior desvio é de {maior:.2f} ponto percentual, {'dentro' if maior <= 3 else 'fora'} da faixa de ±3.

A proporção de positivos não é a mesma nos três conjuntos: {100*t.proporcao_positiva['treino']:.2f}% no treino, {100*t.proporcao_positiva['validacao']:.2f}% na validação ({t.positivos['validacao']} positivos) e {100*t.proporcao_positiva['teste']:.2f}% no teste ({t.positivos['teste']} positivos). Isso vem do sorteio por grupos e eu não mexi na semente para corrigir. A validação tem poucos positivos; vou lembrar disso na escolha do limiar.

Os zeros de perfis compartilhados dizem que nenhuma medida idêntica atravessa a fronteira. Eles não provam que dois vinhos de perfis diferentes são físicos diferentes, nem o contrário.'''))

### B1. Cinco cenários
As respostas dos cinco cenários (avaliações de produtos, turbinas, spam, radiografias e transações) estão no `README.md`, na seção B1, como pede o enunciado. Em todas eu declaro o uso futuro que estou pressupondo e as hipóteses que faltam. No cenário 5, tempo e identidade do cliente são discutidos separadamente.

## A1. Dataset próprio
`MeuDataset` (em `lab02lib/data.py`) recebe uma tabela de uma única partição, as características e, opcionalmente, média e desvio. No treino ele aprende as duas estatísticas só com as próprias linhas. Validação e teste recebem os valores do treino e só aplicam a conta. Convenção: **desvio populacional (`ddof=0`)**, calculado em float64 e guardado em float32. Desvio zero vira 1,0.

Por que isso importa: se a média fosse calculada com a validação ou o teste, a escala do treino já conteria informação desses conjuntos. Essa é a forma 1 de vazamento, que reproduzo na Parte C.

Cada exemplo devolve `X` float32 com forma `[11]` e `y` float32 escalar. Num lote de tamanho B as formas são `[B, 11]` e `[B]`.

In [ ]:
from lab02lib.data import criar_datasets, criar_loaders, tabela_parametros, verificar_datasets

ds_treino, ds_validacao, ds_teste = criar_datasets(particoes, FEATURES)
print('Tamanhos: treino', len(ds_treino), '| validação', len(ds_validacao), '| teste', len(ds_teste))
print('\nDocstrings do MeuDataset:')
for nome in ['__init__', '__len__', '__getitem__']:
    print(f'- {nome}:', ' '.join(getattr(MeuDataset, nome).__doc__.split()))

### A2. Três DataLoaders
O treino embaralha para que cada época veja os exemplos em outra ordem. Isso muda a ordem dentro do treino, não a separação entre as partições. Validação e teste não embaralham e **não descartam nada**: a ordem das previsões precisa continuar alinhada à dos rótulos, e `obter_escores` devolve os dois juntos. As justificativas completas estão na tabela abaixo.

In [ ]:
loader_treino, loader_validacao, loader_teste = criar_loaders(ds_treino, ds_validacao, ds_teste, SEED)
loaders = {'treino': loader_treino, 'validacao': loader_validacao, 'teste': loader_teste}
assert loader_validacao.dataset is ds_validacao and loader_teste.dataset is ds_teste
a2_tabela = tabela_parametros(loaders)
a2_tabela.to_csv(RESULTADOS / 'a2_loaders.csv', index=False)
display(a2_tabela)
a1_estatisticas = verificar_datasets(ds_treino, ds_validacao, ds_teste, loaders, FEATURES)
a1_estatisticas.to_csv(RESULTADOS / 'a1_estatisticas.csv', index=False)
display(a1_estatisticas.round(4))

In [ ]:
e = a1_estatisticas.set_index('conjunto')
display(Markdown(f'''**A1 — como ler.** No treino a maior média absoluta de uma coluna é {e.max_abs_media['treino']:.1e}, praticamente zero, porque as estatísticas saíram dele. Na validação ela é {e.max_abs_media['validacao']:.3f} e no teste {e.max_abs_media['teste']:.3f}. Isso é esperado: validação e teste foram normalizados com a média e o desvio **do treino**, então não precisam ter média zero. O desvio médio por coluna é {e.desvio_medio['validacao']:.3f} na validação e {e.desvio_medio['teste']:.3f} no teste. Se eles tivessem média exatamente zero, eu desconfiaria de que usaram as próprias estatísticas.

**A2 — tabela de decisões.** Os valores abaixo vêm dos carregadores criados.

| Conjunto | batch_size | shuffle | drop_last | num_workers | Efeito da escolha |
|---|---|---|---|---|---|
| treino | {loader_treino.batch_size} | {a2_tabela.shuffle[0]} | {loader_treino.drop_last} | {loader_treino.num_workers} | Lote 64 dá {len(loader_treino)} atualizações por época com {len(ds_treino)} linhas: um meio-termo entre o lote 1 (muitas atualizações ruidosas) e o 512 (poucas atualizações), como a A3 mostra. Embaralhar, com gerador semeado, evita a mesma ordem em toda época e continua reproduzível. Não descarto o último lote (sobra {len(ds_treino) % 64} linha): a rede de referência não tem BatchNorm. |
| validação | {loader_validacao.batch_size} | {a2_tabela.shuffle[1]} | {loader_validacao.drop_last} | {loader_validacao.num_workers} | Sem gradientes, o lote só afeta memória, então cabe a validação inteira ({len(ds_validacao)} linhas) em um lote. Sem embaralhar, a ordem dos escores fica igual à dos rótulos. Nenhuma observação é descartada. |
| teste | {loader_teste.batch_size} | {a2_tabela.shuffle[2]} | {loader_teste.drop_last} | {loader_teste.num_workers} | Mesmo raciocínio da validação ({len(ds_teste)} linhas). O teste só é percorrido na avaliação final. |

`num_workers=0` nos três: os dados já estão em tensores na memória, então não há leitura de disco para esconder. Processos auxiliares custam para abrir e para trocar dados. Eu **não medi** workers maiores que 0, então não afirmo que ajudam ou atrapalham; só escolhi o caminho que não depende de criar processos.'''))

## A3. Tempo de uma época
Compara `batch_size` 1, 8, 64 e 512 com a rede **sem BatchNorm**. Uma medição é uma época completa de `treinar_epoca` (previsão, perda, retropropagação e passo do otimizador) cronometrada com `time.perf_counter()`.

Para a comparação ser justa:
- o estado inicial é guardado uma vez e cada medição reconstrói o modelo a partir dele, com um **Adam novo**;
- cada medição usa um DataLoader novo (o `loader_treino` da Parte D não é tocado);
- há uma época de aquecimento por tamanho, fora da medição;
- são três repetições por tamanho, em rodízio entre os tamanhos, para que a oscilação da máquina não caia sempre no mesmo lote;
- nenhuma medida é descartada.

In [ ]:
from lab02lib.timing import medir_tempos, resumir_tempos, grafico_tempos, descrever_hardware
import json

hardware = descrever_hardware()
tempos = medir_tempos(ds_treino, [1, 8, 64, 512], repeticoes=3, semente=SEED, dispositivo=DISPOSITIVO)
a3_resumo = resumir_tempos(tempos)
pd.DataFrame(tempos).to_csv(RESULTADOS / 'a3_tempos.csv', index=False)
a3_resumo.to_csv(RESULTADOS / 'a3_resumo.csv', index=False)
(RESULTADOS / 'a3_hardware.json').write_text(json.dumps(hardware, indent=2, ensure_ascii=False), encoding='utf-8')
display(pd.DataFrame(tempos).round(5))
display(a3_resumo.round(6))
print(hardware)
display(grafico_tempos(tempos, hardware, RESULTADOS / 'a3_tempo_x_lote.png'))

In [ ]:
r = a3_resumo.set_index('batch_size')
razoes = [r.mediana_segundos[a] / r.mediana_segundos[b] for a, b in [(1, 8), (8, 64), (64, 512)]]
display(Markdown(f'''**Como ler.** A mediana de uma época foi {r.mediana_segundos[1]:.4f} s com lote 1, {r.mediana_segundos[8]:.4f} s com lote 8, {r.mediana_segundos[64]:.4f} s com lote 64 e {r.mediana_segundos[512]:.4f} s com lote 512. Cada degrau multiplicou o lote por 8 e dividiu o tempo por {razoes[0]:.1f}, {razoes[1]:.1f} e {razoes[2]:.1f}. {'O ganho encolheu a cada degrau, mas não chegou a zero.' if razoes[0] > razoes[1] > razoes[2] and razoes[2] > 1.2 else 'A sequência de ganhos não é monotônica ou chegou perto de um platô.'} Se o último ganho ficasse perto de 1, eu diria que o tempo achatou. Não é o que aparece nesta medição.

O custo de **um passo** sobe com o lote: {r.mediana_segundos_por_passo[1]*1000:.3f} ms, {r.mediana_segundos_por_passo[8]*1000:.3f} ms, {r.mediana_segundos_por_passo[64]*1000:.3f} ms e {r.mediana_segundos_por_passo[512]*1000:.3f} ms. Cada passo processa mais linhas, mas o custo não cresce na mesma proporção do lote. Por isso a época fica mais curta com lotes grandes: há menos passos ({r.passos[1]}, {r.passos[8]}, {r.passos[64]} e {r.passos[512]}).

Lotes diferentes também significam **quantidades diferentes de atualizações de pesos por época**. Com lote 1 o otimizador atualiza {r.passos[1]} vezes; com lote 512, {r.passos[512]}. Ir mais rápido por época não quer dizer aprender o mesmo: com poucas atualizações a rede pode precisar de mais épocas. A medição do tempo não diz nada sobre a qualidade do modelo.'''))

## A4. BatchNorm com último lote 1
`nn.BatchNorm1d` em modo `train()` calcula média e variância do lote. Com **uma** única observação não existe variância, e o PyTorch levanta `ValueError`. Isso acontece quando o número de linhas deixa resto 1 na divisão pelo lote: aqui, 65 linhas do treino e lote 64 (65 % 64 = 1).

O experimento usa só 65 linhas do treino (sorteadas com semente) e `criar_modelo(11, batchnorm=True)` com `drop_last=False`. Capturo **somente** o `ValueError` com a mensagem esperada. Qualquer outro erro continua subindo. Depois, com um **modelo novo e a mesma semente** (o primeiro já deu um passo e ficou contaminado), uso a correção `drop_last=True` com `shuffle=True`. Para mostrar qual linha foi descartada, um carregador espelho com a mesma semente devolve também o `id_linha`; ele só serve para a demonstração.

In [ ]:
from lab02lib.batchnorm import demonstrar_batchnorm

a4_log = demonstrar_batchnorm(ds_treino, RESULTADOS / 'a4_log.txt', semente=SEED)

**Como ler o A4.** O erro aparece no passo do último lote, que tem uma observação. A correção descarta esse resto, e a linha descartada **muda a cada época** porque o treino embaralha antes de cortar o lote incompleto. Por isso o custo é pequeno: uma linha por época e nunca a mesma. Sem embaralhar, a mesma linha ficaria sempre de fora.

Isso vale só para o treino. Na validação e no teste eu **não** uso `drop_last=True`: cada observação precisa ser avaliada (o `apoio.py` até recusa `drop_last=True` na avaliação). E o modelo avaliado precisa estar em `eval()`, para o BatchNorm usar as estatísticas acumuladas no treino e não as do lote. O modelo principal desta entrega não usa BatchNorm; a variante só existe aqui.